# Assault DDQN - HU007 End-to-end smoke


## 1. Bootstrap Local -> GitHub -> Colab

In [1]:
import os


os.environ["ASSAULT_BOOTSTRAP_REF"] = "fix/hu007-release-gpu-memory"
os.environ["ASSAULT_E2E_REQUIRE_CUDA"] = "1"

In [2]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


Execution bootstrap
  runtime: Google Colab
  repository: /content/reinforcement_learning_reto_1
  assault_dir: /content/reinforcement_learning_reto_1/2_Assault
  requested_ref: fix/hu007-release-gpu-memory
  requested_commit: <none>
  resolved_sha: eb72de9e9208f2ac8cdb39048b073bb4ddfe4c98
  requirements: /content/reinforcement_learning_reto_1/2_Assault/requirements.txt
src.environment import source: /content/reinforcement_learning_reto_1/2_Assault/src/environment.py


{'is_colab': True,
 'repo_root': '/content/reinforcement_learning_reto_1',
 'assault_dir': '/content/reinforcement_learning_reto_1/2_Assault',
 'requested_ref': 'fix/hu007-release-gpu-memory',
 'requested_commit': None,
 'resolved_sha': 'eb72de9e9208f2ac8cdb39048b073bb4ddfe4c98',
 'requirements_path': '/content/reinforcement_learning_reto_1/2_Assault/requirements.txt',
 'environment_source': '/content/reinforcement_learning_reto_1/2_Assault/src/environment.py'}

In [3]:
from dataclasses import fields
from src.e2e_smoke import E2ESmokeSummary
import src.e2e_smoke as e2e_smoke_module

print(e2e_smoke_module.__file__)
print([field.name for field in fields(E2ESmokeSummary)])

/content/reinforcement_learning_reto_1/2_Assault/src/e2e_smoke.py
['local_e2e_smoke_pass', 'e2e_smoke_pass', 'runtime', 'device', 'require_cuda', 'run_id', 'observation_shape', 'observation_dtype', 'action_space', 'preflight', 'segment_a', 'checkpoint', 'restored', 'segment_b', 'tensorboard_event_files_before', 'tensorboard_event_files_after', 'tensorboard_tags', 'tensorboard_steps_before', 'tensorboard_steps_after', 'tensorboard_post_resume_steps', 'tensorboard_previous_logs_preserved', 'evaluation', 'online_unchanged_during_evaluation', 'target_unchanged_during_evaluation', 'optimizer_unchanged_during_evaluation', 'replay_buffer_unchanged_during_evaluation', 'training_global_step_unchanged_during_evaluation', 'memory_before', 'memory_after_segment_a', 'memory_after_release', 'memory_after', 'runtime_info', 'duration_seconds']


## 2. Imports and configuration

In [4]:
from pathlib import Path

from src.agent import DDQNAgent
from src.callbacks import TensorBoardLogger, load_tensorboard_scalars
from src.checkpointing import CheckpointManager, reconstruct_epsilon
from src.e2e_smoke import run_e2e_smoke
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.preflight import run_preflight_checks
from src.replay_buffer import ReplayBuffer
from src.trainer import Trainer
from src.utils import get_runtime_info, load_yaml_config

config = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
config


PROJECT_ROOT: /content/reinforcement_learning_reto_1
ASSAULT_DIR: /content/reinforcement_learning_reto_1/2_Assault
BOOTSTRAP_REF: fix/hu007-release-gpu-memory
BOOTSTRAP_COMMIT: <none>
EXECUTED_SHA: eb72de9e9208f2ac8cdb39048b073bb4ddfe4c98
src.environment: /content/reinforcement_learning_reto_1/2_Assault/src/environment.py


{'environment': {'id': 'ALE/Assault-v5',
  'obs_type': 'rgb',
  'frame_skip': 4,
  'repeat_action_probability': 0.25,
  'full_action_space': False,
  'render_mode': None},
 'preprocessing': {'grayscale': True,
  'resize_height': 84,
  'resize_width': 84,
  'frame_stack': 4,
  'dtype': 'uint8',
  'normalize_pixels_in_env': False},
 'reproducibility': {'seed': 42},
 'evaluation': {'episodes': 10},
 'network': {'input_channels': 4, 'num_actions': 7},
 'agent': {'gamma': 0.99,
  'learning_rate': 0.0001,
  'epsilon_start': 1.0,
  'epsilon_final': 0.01},
 'replay_buffer': {'capacity': 1024, 'batch_size': 32},
 'training': {'total_timesteps': 48,
  'learning_starts': 32,
  'train_frequency': 4,
  'target_update_frequency': 16,
  'epsilon_decay_steps': 48},
 'checkpointing': {'enabled': True,
  'interval_steps': 24,
  'directory': 'checkpoints',
  'mode': 'new',
  'run_id': 'assault_ddqn_exp_001',
  'resume_checkpoint': None,
  'save_replay_buffer': True},
 'tensorboard': {'enabled': True,
  '

## 3. Runtime and hardware

In [5]:
runtime_info = get_runtime_info()
runtime_info


{'python_version': '3.13.15',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'gymnasium_version': '1.1.1',
 'ale_py_version': '0.10.1',
 'cpu': 'x86_64',
 'cpu_count_logical': 12,
 'cpu_count_physical': 6,
 'ram_total_gb': 83.47,
 'ram_available_gb': 81.33,
 'gpu_available': True,
 'gpu_name': 'NVIDIA A100-SXM4-40GB',
 'gpu_vram_total_gb': 39.49,
 'torch_version': '2.11.0+cu128'}

## 4. HU002 environment contract

In [6]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


Observation shape: (4, 84, 84)
Observation dtype: uint8
Action space: Discrete(7)
Action meanings: ['NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']
Initial info: {'lives': 4, 'episode_frame_number': 0, 'frame_number': 0, 'seeds': (np.uint32(3444837047), np.uint32(2669555309))}
Metadata: EnvironmentMetadata(env_id='ALE/Assault-v5', mode='train', seed=42, action_space='Discrete(7)', action_meanings=('NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE'), observation_shape=(4, 84, 84), observation_dtype='uint8', base_frameskip=4, wrapper_frameskip=1, effective_frameskip=4, repeat_action_probability=0.25, full_action_space=False)


## 5. HU002 autovalidations

In [7]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


HU002 validations passed.


## 6. HU004 preflight gate

In [8]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


===== DDQN PRE-FLIGHT =====
Runtime: Google Colab
Device: cuda
Device: PASS
Environment: PASS
Observation: PASS (4, 84, 84) uint8
QNetwork: PASS -> (1, 7)
ReplayBuffer: PASS
DDQN update: PASS loss=0.000926
Loss finite: PASS
Target stable: PASS
Target sync: PASS
Save/load: PASS temporary_file_cleaned=True

READY_FOR_TRAINING=True


{'passed': True,
 'ready_for_training': True,
 'runtime': 'Google Colab',
 'device': 'cuda',
 'checks': {'Device': True,
  'Environment': True,
  'Observation': True,
  'QNetwork': True,
  'ReplayBuffer': True,
  'DDQN update': True,
  'Loss finite': True,
  'Target stable': True,
  'Target sync': True,
  'Save/load': True},
 'errors': [],
 'details': {'runtime_info': {'python_version': '3.13.15',
   'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
   'gymnasium_version': '1.1.1',
   'ale_py_version': '0.10.1',
   'cpu': 'x86_64',
   'cpu_count_logical': 12,
   'cpu_count_physical': 6,
   'ram_total_gb': 83.47,
   'ram_available_gb': 81.27,
   'gpu_available': True,
   'gpu_name': 'NVIDIA A100-SXM4-40GB',
   'gpu_vram_total_gb': 39.49,
   'torch_version': '2.11.0+cu128'},
  'Observation': '(4, 84, 84) uint8',
  'QNetwork': '-> (1, 7)',
  'DDQN update': 'loss=0.000926',
  'Save/load': 'temporary_file_cleaned=True'}}

## 7. Abort if preflight fails

In [9]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU005 training aborted.")
print("READY_FOR_TRAINING=True")


READY_FOR_TRAINING=True


## 8. HU007 E2E smoke configuration


In [10]:
def _env_flag(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y"}

e2e_config = config.get("e2e_smoke", {})
RUN_ID = os.environ.get("ASSAULT_RUN_ID", config["checkpointing"]["run_id"] + "_hu007_smoke")
CHECKPOINT_DIR = Path(os.environ.get("ASSAULT_CHECKPOINT_DIR", str(ASSAULT_DIR / config["checkpointing"]["directory"])))
TENSORBOARD_DIR = Path(os.environ.get("ASSAULT_TENSORBOARD_DIR", str(ASSAULT_DIR / config.get("tensorboard", {}).get("directory", "logs/tensorboard"))))
E2E_REQUIRE_CUDA = _env_flag("ASSAULT_E2E_REQUIRE_CUDA", bool(e2e_config.get("require_cuda", True)))

print("HU007 run_id:", RUN_ID)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("TensorBoard directory:", TENSORBOARD_DIR)
print("Segment A timesteps:", e2e_config.get("segment_a_timesteps"))
print("Final timesteps:", e2e_config.get("final_timesteps"))
print("Evaluation episodes:", e2e_config.get("evaluation_episodes"))
print("Evaluation epsilon:", e2e_config.get("evaluation_epsilon"))
print("Require CUDA:", E2E_REQUIRE_CUDA)


HU007 run_id: assault_ddqn_exp_001_hu007_smoke
Checkpoint directory: /content/reinforcement_learning_reto_1/2_Assault/checkpoints
TensorBoard directory: /content/reinforcement_learning_reto_1/2_Assault/logs/tensorboard
Segment A timesteps: 48
Final timesteps: 64
Evaluation episodes: 2
Evaluation epsilon: 0.0
Require CUDA: True


## 9. HU007 end-to-end smoke


In [11]:
e2e_summary = run_e2e_smoke(
    config=config,
    checkpoint_root=CHECKPOINT_DIR,
    tensorboard_root=TENSORBOARD_DIR,
    run_id=RUN_ID,
    repo_path=PROJECT_ROOT,
    require_cuda=E2E_REQUIRE_CUDA,
)
e2e_result = e2e_summary.as_dict()
e2e_result


{'LOCAL_E2E_SMOKE_PASS': False,
 'E2E_SMOKE_PASS': True,
 'runtime': 'Google Colab',
 'device': 'cuda',
 'require_cuda': True,
 'run_id': 'assault_ddqn_exp_001_hu007_smoke',
 'observation_shape': (4, 84, 84),
 'observation_dtype': 'uint8',
 'action_space': 'Discrete(7)',
 'preflight': {'passed': True,
  'ready_for_training': True,
  'runtime': 'Google Colab',
  'device': 'cuda',
  'checks': {'Device': True,
   'Environment': True,
   'Observation': True,
   'QNetwork': True,
   'ReplayBuffer': True,
   'DDQN update': True,
   'Loss finite': True,
   'Target stable': True,
   'Target sync': True,
   'Save/load': True},
  'errors': [],
  'details': {'runtime_info': {'python_version': '3.13.15',
    'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
    'gymnasium_version': '1.1.1',
    'ale_py_version': '0.10.1',
    'cpu': 'x86_64',
    'cpu_count_logical': 12,
    'cpu_count_physical': 6,
    'ram_total_gb': 83.47,
    'ram_available_gb': 80.56,
    'gpu_available': True,
    'gpu_nam

## 10. HU007 result gates


In [12]:
assert e2e_summary.preflight.ready_for_training
assert e2e_summary.segment_a.global_step == int(e2e_config["segment_a_timesteps"])
assert e2e_summary.restored.global_step == e2e_summary.checkpoint.checkpoint_step
assert e2e_summary.segment_b.global_step == int(e2e_config["final_timesteps"])
assert e2e_summary.tensorboard_previous_logs_preserved
assert e2e_summary.online_unchanged_during_evaluation
assert e2e_summary.target_unchanged_during_evaluation
assert e2e_summary.optimizer_unchanged_during_evaluation
assert e2e_summary.replay_buffer_unchanged_during_evaluation
assert e2e_summary.training_global_step_unchanged_during_evaluation

if e2e_summary.runtime == "Google Colab" and E2E_REQUIRE_CUDA:
    assert e2e_summary.e2e_smoke_pass, "Colab GPU smoke did not pass."
else:
    assert not e2e_summary.e2e_smoke_pass, "E2E_SMOKE_PASS must remain false outside Colab GPU."

print("HU007 E2E smoke local status")
print("runtime:", e2e_summary.runtime)
print("device:", e2e_summary.device)
print("run_id:", e2e_summary.run_id)
print("observation:", e2e_summary.observation_shape, e2e_summary.observation_dtype)
print("action_space:", e2e_summary.action_space)
print("Preflight READY_FOR_TRAINING:", e2e_summary.preflight.ready_for_training)
print("segment_a_steps:", e2e_summary.segment_a.global_step)
print("segment_a_updates:", e2e_summary.segment_a.updates_count)
print("segment_a_last_loss:", e2e_summary.segment_a.last_loss)
print("segment_a_last_q_mean:", e2e_summary.segment_a.last_q_mean)
print("checkpoint_path:", e2e_summary.checkpoint.path)
print("checkpoint_size_bytes:", e2e_summary.checkpoint.size_bytes)
print("restored_global_step:", e2e_summary.restored.global_step)
print("replay_buffer_restored:", e2e_summary.restored.replay_buffer_restored)
print("segment_b_final_step:", e2e_summary.segment_b.global_step)
print("segment_b_updates_total:", e2e_summary.segment_b.updates_count)
print("tensorboard_event_files:", e2e_summary.tensorboard_event_files_after)
print("tensorboard_tags:", e2e_summary.tensorboard_tags)
print("tensorboard_post_resume_steps:", e2e_summary.tensorboard_post_resume_steps)
print("evaluation_rewards:", e2e_summary.evaluation.rewards)
print("evaluation_mean_reward:", e2e_summary.evaluation.mean_reward)
print("evaluation_lengths:", e2e_summary.evaluation.episode_lengths)
print("memory_before:", e2e_summary.memory_before.as_dict())
print("memory_after_segment_a:", e2e_summary.memory_after_segment_a.as_dict())
print("memory_after_release:", e2e_summary.memory_after_release.as_dict())
print("memory_after:", e2e_summary.memory_after.as_dict())
print("LOCAL_E2E_SMOKE_PASS=", e2e_summary.local_e2e_smoke_pass, sep="")
print("E2E_SMOKE_PASS=", e2e_summary.e2e_smoke_pass, sep="")
if not e2e_summary.e2e_smoke_pass:
    print("E2E_SMOKE_PASS final queda pendiente de ejecucion real en Colab GPU.")
print("Local TensorBoard command:", "tensorboard --logdir " + str(TENSORBOARD_DIR))
if bootstrap.is_colab:
    print("Colab commands: %load_ext tensorboard ; %tensorboard --logdir " + str(TENSORBOARD_DIR))


HU007 E2E smoke local status
runtime: Google Colab
device: cuda
run_id: assault_ddqn_exp_001_hu007_smoke
observation: (4, 84, 84) uint8
action_space: Discrete(7)
Preflight READY_FOR_TRAINING: True
segment_a_steps: 48
segment_a_updates: 5
segment_a_last_loss: 0.0008681190083734691
segment_a_last_q_mean: 0.028299855068325996
checkpoint_path: /content/reinforcement_learning_reto_1/2_Assault/checkpoints/assault_ddqn_exp_001_hu007_smoke/checkpoint_step_000048.pt
checkpoint_size_bytes: 29742069
restored_global_step: 48
replay_buffer_restored: True
segment_b_final_step: 64
segment_b_updates_total: 9
tensorboard_event_files: 2
tensorboard_tags: ['train/epsilon', 'train/learning_rate', 'train/loss', 'train/q_mean']
tensorboard_post_resume_steps: {'train/epsilon': [52, 56, 60, 64], 'train/loss': [52, 56, 60, 64], 'train/q_mean': [52, 56, 60, 64], 'train/learning_rate': [52, 56, 60, 64]}
evaluation_rewards: [0.0, 0.0]
evaluation_mean_reward: 0.0
evaluation_lengths: [256, 256]
memory_before: {'lab